# Convolution Comparison: scipy vs astropy vs `_nanconvolve`

This notebook compares three convolution approaches for handling NaN and masked pixels:

1. **scipy** (`scipy.ndimage.convolve`) — NaN values propagate through the output
2. **astropy** (`astropy.convolution.convolve` with `nan_treatment='interpolate'`) — interpolates over NaN for normalized kernels, but fails for zero-sum kernels
3. **`_nanconvolve`** — kernel-decomposition approach that works for both normalized and zero-sum kernels

We test with four data scenarios × two kernel types (normalized Gaussian and zero-sum DAO-like).

## 1. Import Required Libraries

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.convolution import Gaussian2DKernel, convolve
from scipy.ndimage import convolve as ndi_convolve

from photutils.utils._convolution import _nanconvolve

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

## 2. Define Kernels (Normalized and Zero-Sum)

In [ ]:
# Normalized Gaussian kernel (sum = 1)
gauss_kernel = Gaussian2DKernel(2, x_size=9, y_size=9)
kernel_norm = gauss_kernel.array

# Zero-sum kernel: DAO-like detection kernel
# Gaussian core minus constant background (sums to ~0)
kernel_zerosum = kernel_norm - kernel_norm.mean()
print(f'Normalized kernel sum: {kernel_norm.sum():.6f}')
print(f'Zero-sum kernel sum:   {kernel_zerosum.sum():.2e}')

# Visualize the kernels
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, k, title in zip(axes,
                         [kernel_norm, kernel_zerosum],
                         ['Normalized Gaussian (sum=1)',
                          'Zero-Sum DAO-like (sum≈0)']):
    im = ax.imshow(k, origin='lower', cmap='RdBu_r',
                   vmin=-np.max(np.abs(k)), vmax=np.max(np.abs(k)))
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.85)
fig.tight_layout()
plt.show()

## 3. Define Test Cases

Four scenarios with increasing difficulty:
1. **Scattered NaN** — 50 random NaN pixels
2. **Mask only** — 50 random masked pixels (no NaN in data)
3. **NaN + mask** — 30 NaN pixels and 30 masked pixels (partially overlapping)
4. **Large NaN cluster** — a contiguous 10×10 NaN block (exceeds kernel footprint)

In [ ]:
rng = np.random.default_rng(42)
base_data = rng.standard_normal((50, 50))

# Test 1: Scattered NaN
data_nan = base_data.copy()
nan_idx = rng.choice(base_data.size, size=50, replace=False)
data_nan.ravel()[nan_idx] = np.nan
mask_nan = None

# Test 2: Mask only (no NaN in data)
data_maskonly = base_data.copy()
mask_maskonly = np.zeros(base_data.shape, dtype=bool)
mask_idx = rng.choice(base_data.size, size=50, replace=False)
mask_maskonly.ravel()[mask_idx] = True

# Test 3: NaN + mask combined
data_both = base_data.copy()
nan_idx2 = rng.choice(base_data.size, size=30, replace=False)
data_both.ravel()[nan_idx2] = np.nan
mask_both = np.zeros(base_data.shape, dtype=bool)
mask_idx2 = rng.choice(base_data.size, size=30, replace=False)
mask_both.ravel()[mask_idx2] = True

# Test 4: Large NaN cluster
data_cluster = base_data.copy()
data_cluster[20:30, 20:30] = np.nan
mask_cluster = None

test_cases = [
    {'name': 'Scattered NaN (50 pixels)',
     'data': data_nan, 'mask': mask_nan},
    {'name': 'Mask only (50 pixels)',
     'data': data_maskonly, 'mask': mask_maskonly},
    {'name': 'NaN + mask (30+30 pixels)',
     'data': data_both, 'mask': mask_both},
    {'name': 'Large NaN cluster (10×10)',
     'data': data_cluster, 'mask': mask_cluster},
]

# Show the invalid regions for each test case
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, tc in zip(axes, test_cases):
    invalid = np.isnan(tc['data'])
    if tc['mask'] is not None:
        invalid = invalid | tc['mask']
    ax.imshow(invalid, origin='lower', cmap='Reds', vmin=0, vmax=1)
    ax.set_title(tc['name'], fontsize=9)
    ax.set_xlabel(f'{np.sum(invalid)} invalid pixels', fontsize=8)
fig.suptitle('Invalid pixel locations for each test case', fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

## 4. Convolution Wrappers

Thin wrappers around each backend so they accept the same inputs.

- **scipy**: Replaces NaN/masked pixels with 0, then convolves. NaN propagates naturally.
- **astropy**: Sets masked pixels to NaN, uses `nan_treatment='interpolate'` for normalized kernels, `nan_treatment='fill'` + `normalize_kernel=False` for zero-sum kernels.
- **`_nanconvolve`**: Handles NaN and mask natively via kernel decomposition.

In [ ]:
def convolve_scipy(data, kernel, mask=None):
    """scipy.ndimage.convolve — NaN propagates through the output."""
    d = data.copy()
    if mask is not None:
        d[mask] = np.nan
    return ndi_convolve(d, kernel, mode='constant', cval=0.0)


def convolve_astropy(data, kernel, mask=None, zero_sum=False):
    """astropy.convolution.convolve with NaN handling.

    For normalized kernels: nan_treatment='interpolate'.
    For zero-sum kernels: nan_treatment='fill' (replace NaN with 0),
    normalize_kernel=False. This is the best astropy can do for
    zero-sum kernels.
    """
    d = data.copy()
    if mask is not None:
        d[mask] = np.nan

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        if zero_sum:
            # astropy cannot renormalize zero-sum kernels with
            # nan_treatment='interpolate' — it raises an error.
            # Best alternative: fill NaN with 0 and do not normalize.
            result = convolve(d, kernel, nan_treatment='fill',
                              normalize_kernel=False,
                              boundary='fill', fill_value=0.0)
        else:
            result = convolve(d, kernel, nan_treatment='interpolate',
                              boundary='fill', fill_value=0.0)
    return result


def convolve_nanconv(data, kernel, mask=None):
    """_nanconvolve — kernel decomposition approach."""
    return _nanconvolve(data, kernel, mask=mask, mode='constant',
                        fill_value=0.0)

## 5. Run Convolutions — Normalized Kernel

In [ ]:
# Ground truth: convolve the clean base_data (no NaN/mask)
gt_norm = ndi_convolve(base_data, kernel_norm, mode='constant', cval=0.0)

results_norm = {}
for tc in test_cases:
    name = tc['name']
    d, m = tc['data'], tc['mask']
    results_norm[name] = {
        'scipy': convolve_scipy(d, kernel_norm, mask=m),
        'astropy': convolve_astropy(d, kernel_norm, mask=m,
                                    zero_sum=False),
        '_nanconvolve': convolve_nanconv(d, kernel_norm, mask=m),
    }
    # Print quick stats
    for impl, r in results_norm[name].items():
        n_nan = np.sum(np.isnan(r))
        valid = ~np.isnan(r) & ~np.isnan(gt_norm)
        if valid.any():
            maxerr = np.max(np.abs(r[valid] - gt_norm[valid]))
        else:
            maxerr = np.nan
        print(f'  [{name}] {impl:15s}: '
              f'NaN={n_nan:4d}, max|err|={maxerr:.3e}')
    print()

## 6. Run Convolutions — Zero-Sum Kernel

In [ ]:
# Ground truth: convolve the clean base_data with the zero-sum kernel
gt_zero = ndi_convolve(base_data, kernel_zerosum, mode='constant', cval=0.0)

results_zero = {}
for tc in test_cases:
    name = tc['name']
    d, m = tc['data'], tc['mask']
    results_zero[name] = {
        'scipy': convolve_scipy(d, kernel_zerosum, mask=m),
        'astropy': convolve_astropy(d, kernel_zerosum, mask=m,
                                    zero_sum=True),
        '_nanconvolve': convolve_nanconv(d, kernel_zerosum, mask=m),
    }
    for impl, r in results_zero[name].items():
        n_nan = np.sum(np.isnan(r))
        valid = ~np.isnan(r) & ~np.isnan(gt_zero)
        if valid.any():
            maxerr = np.max(np.abs(r[valid] - gt_zero[valid]))
        else:
            maxerr = np.nan
        print(f'  [{name}] {impl:15s}: '
              f'NaN={n_nan:4d}, max|err|={maxerr:.3e}')
    print()

## 7. Visualize — Normalized Kernel

Each row is one test case. Columns: input data, scipy result, astropy result, `_nanconvolve` result.

In [ ]:
impl_names = ['scipy', 'astropy', '_nanconvolve']

fig, axes = plt.subplots(len(test_cases), 5, figsize=(18, 3.5 * len(test_cases)))

for i, tc in enumerate(test_cases):
    name = tc['name']
    vmin, vmax = np.nanpercentile(base_data, [2, 98])

    # Input data
    ax = axes[i, 0]
    im = ax.imshow(tc['data'], origin='lower', cmap='viridis',
                   vmin=vmin, vmax=vmax)
    ax.set_title(f'Input: {name}', fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Ground truth
    ax = axes[i, 1]
    im = ax.imshow(gt_norm, origin='lower', cmap='viridis',
                   vmin=vmin, vmax=vmax)
    ax.set_title('Ground truth', fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Three implementations
    for j, impl in enumerate(impl_names):
        ax = axes[i, j + 2]
        r = results_norm[name][impl]
        n_nan = np.sum(np.isnan(r))
        im = ax.imshow(r, origin='lower', cmap='viridis',
                       vmin=vmin, vmax=vmax)
        label = f'{impl}'
        if n_nan > 0:
            label += f' ({n_nan} NaN)'
        ax.set_title(label, fontsize=9)
        plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Normalized Kernel — Convolution Results', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 8. Visualize — Zero-Sum Kernel

In [ ]:
fig, axes = plt.subplots(len(test_cases), 5, figsize=(18, 3.5 * len(test_cases)))

for i, tc in enumerate(test_cases):
    name = tc['name']
    vmin_z, vmax_z = np.nanpercentile(gt_zero, [2, 98])

    # Input data
    ax = axes[i, 0]
    vmin_d, vmax_d = np.nanpercentile(base_data, [2, 98])
    im = ax.imshow(tc['data'], origin='lower', cmap='viridis',
                   vmin=vmin_d, vmax=vmax_d)
    ax.set_title(f'Input: {name}', fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Ground truth
    ax = axes[i, 1]
    im = ax.imshow(gt_zero, origin='lower', cmap='RdBu_r',
                   vmin=-np.max(np.abs(gt_zero)),
                   vmax=np.max(np.abs(gt_zero)))
    ax.set_title('Ground truth', fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

    # Three implementations
    for j, impl in enumerate(impl_names):
        ax = axes[i, j + 2]
        r = results_zero[name][impl]
        n_nan = np.sum(np.isnan(r))
        maxabs = np.nanmax(np.abs(gt_zero))
        im = ax.imshow(r, origin='lower', cmap='RdBu_r',
                       vmin=-maxabs, vmax=maxabs)
        label = f'{impl}'
        if n_nan > 0:
            label += f' ({n_nan} NaN)'
        ax.set_title(label, fontsize=9)
        plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Zero-Sum Kernel — Convolution Results', fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

## 9. Difference Maps (Error vs Ground Truth)

For each test case + kernel type, we show the error (result − ground truth) for all three methods. NaN pixels in the result are shown in gray.

In [ ]:
def plot_error_maps(results, gt, kernel_label, test_cases):
    """Plot error = result - ground_truth for each test case × method."""
    fig, axes = plt.subplots(len(test_cases), 3,
                             figsize=(13, 3.5 * len(test_cases)))

    for i, tc in enumerate(test_cases):
        name = tc['name']
        for j, impl in enumerate(impl_names):
            ax = axes[i, j]
            r = results[name][impl]
            err = r - gt

            # Compute max absolute error over valid pixels
            valid = ~np.isnan(err)
            if valid.any():
                maxerr = np.nanmax(np.abs(err[valid]))
                meanerr = np.nanmean(np.abs(err[valid]))
            else:
                maxerr = meanerr = np.nan

            # Symmetric color scale
            vlim = max(np.nanmax(np.abs(err)) if valid.any() else 1, 1e-10)
            cmap = plt.cm.RdBu_r.copy()
            cmap.set_bad('0.5')  # gray for NaN
            im = ax.imshow(err, origin='lower', cmap=cmap,
                           vmin=-vlim, vmax=vlim)
            ax.set_title(f'{impl}\nmax|err|={maxerr:.2e}', fontsize=8)
            plt.colorbar(im, ax=ax, shrink=0.8)

            if j == 0:
                ax.set_ylabel(name, fontsize=9)

    fig.suptitle(f'{kernel_label} — Error vs Ground Truth',
                 fontsize=14, y=1.01)
    fig.tight_layout()
    plt.show()


plot_error_maps(results_norm, gt_norm, 'Normalized Kernel', test_cases)

In [ ]:
plot_error_maps(results_zero, gt_zero, 'Zero-Sum Kernel', test_cases)

## 10. Summary Statistics Table

For each kernel type × test case × method, we compute:
- **# NaN** in the output
- **Max |error|** vs ground truth
- **Mean |error|** vs ground truth
- **RMS error** vs ground truth

In [ ]:
rows = []

for kernel_label, results, gt in [
    ('Normalized', results_norm, gt_norm),
    ('Zero-Sum', results_zero, gt_zero),
]:
    for tc in test_cases:
        name = tc['name']
        for impl in impl_names:
            r = results[name][impl]
            err = r - gt
            valid = ~np.isnan(err)
            n_nan = int(np.sum(np.isnan(r)))
            if valid.any():
                max_err = np.max(np.abs(err[valid]))
                mean_err = np.mean(np.abs(err[valid]))
                rms_err = np.sqrt(np.mean(err[valid] ** 2))
            else:
                max_err = mean_err = rms_err = np.nan

            rows.append({
                'Kernel': kernel_label,
                'Test Case': name,
                'Method': impl,
                '# NaN': n_nan,
                'Max |err|': max_err,
                'Mean |err|': mean_err,
                'RMS err': rms_err,
            })

df = pd.DataFrame(rows)

# Format for display
styled = (df.style
          .format({
              'Max |err|': '{:.3e}',
              'Mean |err|': '{:.3e}',
              'RMS err': '{:.3e}',
          })
          .set_caption('Convolution Error Summary vs Ground Truth')
          .set_table_styles([
              {'selector': 'caption',
               'props': [('font-size', '14px'), ('font-weight', 'bold')]},
          ])
          )
styled

## 11. Reinsert NaN at Invalid Pixels and Recompute Statistics

In practice, after convolution one would mark the originally-invalid pixels as NaN in the output (the convolved values at those locations are unreliable). Here we:

1. Take each convolution result and set originally-NaN and originally-masked pixels back to NaN.
2. Recompute the error statistics **only over valid (non-NaN, non-masked) pixels**.
3. Visualize the "NaN-reinserted" results and error maps side by side.

In [ ]:
def reinsert_nan(result, data, mask):
    """Set originally-invalid pixels back to NaN in the convolution result."""
    out = result.copy()
    invalid = np.isnan(data)
    if mask is not None:
        invalid = invalid | mask
    out[invalid] = np.nan
    return out


# Build NaN-reinserted results for both kernel types
results_norm_nan = {}
results_zero_nan = {}

for tc in test_cases:
    name = tc['name']
    d, m = tc['data'], tc['mask']

    results_norm_nan[name] = {}
    results_zero_nan[name] = {}
    for impl in impl_names:
        results_norm_nan[name][impl] = reinsert_nan(
            results_norm[name][impl], d, m)
        results_zero_nan[name][impl] = reinsert_nan(
            results_zero[name][impl], d, m)

print('NaN-reinserted results computed.')

### Normalized Kernel — NaN-reinserted results and error maps

In [ ]:
def plot_reinserted(results_ri, gt, kernel_label, test_cases):
    """Show NaN-reinserted results + error maps for each test case."""
    n_tc = len(test_cases)
    fig, axes = plt.subplots(n_tc, 4, figsize=(16, 3.5 * n_tc))

    cmap_err = plt.cm.RdBu_r.copy()
    cmap_err.set_bad('0.5')
    cmap_data = plt.cm.viridis.copy()
    cmap_data.set_bad('0.5')

    for i, tc in enumerate(test_cases):
        name = tc['name']

        # Column 0: ground truth with invalid pixels shown as gray
        ax = axes[i, 0]
        gt_display = gt.copy()
        invalid = np.isnan(tc['data'])
        if tc['mask'] is not None:
            invalid = invalid | tc['mask']
        gt_display[invalid] = np.nan
        vmin, vmax = np.nanpercentile(gt, [2, 98])
        im = ax.imshow(gt_display, origin='lower', cmap=cmap_data,
                       vmin=vmin, vmax=vmax)
        n_inv = int(np.sum(invalid))
        ax.set_title(f'GT (NaN reinserted)\n{n_inv} invalid → gray',
                     fontsize=8)
        plt.colorbar(im, ax=ax, shrink=0.8)
        if i == 0:
            ax.set_ylabel('', fontsize=1)
        ax.set_ylabel(name, fontsize=9)

        # Columns 1-3: three methods
        for j, impl in enumerate(impl_names):
            ax = axes[i, j + 1]
            r = results_ri[name][impl]
            err = r - gt
            valid = ~np.isnan(err)
            n_nan = int(np.sum(np.isnan(r)))
            if valid.any():
                maxerr = np.max(np.abs(err[valid]))
            else:
                maxerr = np.nan
            vlim = max(np.nanmax(np.abs(err[valid])) if valid.any()
                       else 1, 1e-10)
            im = ax.imshow(err, origin='lower', cmap=cmap_err,
                           vmin=-vlim, vmax=vlim)
            ax.set_title(f'{impl}\nmax|err|={maxerr:.2e}  '
                         f'({n_nan} NaN)', fontsize=8)
            plt.colorbar(im, ax=ax, shrink=0.8)

    fig.suptitle(f'{kernel_label} — Error After NaN Reinsertion '
                 f'(gray = invalid)', fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()


plot_reinserted(results_norm_nan, gt_norm, 'Normalized Kernel', test_cases)

### Zero-Sum Kernel — NaN-reinserted results and error maps

In [ ]:
plot_reinserted(results_zero_nan, gt_zero, 'Zero-Sum Kernel', test_cases)

### Summary Table — Statistics over valid pixels only (NaN reinserted)

In [ ]:
rows_ri = []

for kernel_label, results_ri, gt in [
    ('Normalized', results_norm_nan, gt_norm),
    ('Zero-Sum', results_zero_nan, gt_zero),
]:
    for tc in test_cases:
        name = tc['name']
        for impl in impl_names:
            r = results_ri[name][impl]
            err = r - gt

            # Only count valid (non-NaN) pixels in the output
            valid = ~np.isnan(r)
            n_nan = int(np.sum(~valid))
            n_valid = int(np.sum(valid))

            if valid.any():
                abs_err = np.abs(err[valid])
                max_err = np.max(abs_err)
                mean_err = np.mean(abs_err)
                rms_err = np.sqrt(np.mean(err[valid] ** 2))
            else:
                max_err = mean_err = rms_err = np.nan

            rows_ri.append({
                'Kernel': kernel_label,
                'Test Case': name,
                'Method': impl,
                '# Valid': n_valid,
                '# NaN': n_nan,
                'Max |err|': max_err,
                'Mean |err|': mean_err,
                'RMS err': rms_err,
            })

df_ri = pd.DataFrame(rows_ri)

styled_ri = (df_ri.style
             .format({
                 'Max |err|': '{:.3e}',
                 'Mean |err|': '{:.3e}',
                 'RMS err': '{:.3e}',
             })
             .set_caption('Error Summary (NaN reinserted, stats over '
                          'valid pixels only)')
             .set_table_styles([
                 {'selector': 'caption',
                  'props': [('font-size', '14px'),
                            ('font-weight', 'bold')]},
             ])
             )
styled_ri

## 12. New `_nanconvolve` Keywords: `mask_output` and `max_invalid_fraction`

`_nanconvolve` now supports two optional keywords:

- **`mask_output=True`**: Reapply NaN at originally-invalid pixel locations (NaN in data or `True` in mask) in the output. This is useful when you don't want the interpolated values at those locations.

- **`max_invalid_fraction`**: A float between 0 and 1. Output pixels whose kernel footprint overlaps with more than this fraction of invalid pixels are set to NaN. This marks unreliable boundary pixels near large NaN/masked regions. Setting this implicitly enables `mask_output=True`.

Below we demonstrate both features using the large NaN cluster test case with the zero-sum kernel.

### 12a. `mask_output=True` — reapply NaN at invalid locations

In [ ]:
# Use the large NaN cluster test case with both kernel types
data_cluster = base_data.copy()
data_cluster[20:30, 20:30] = np.nan

# Default: interpolates over NaN
r_default = _nanconvolve(data_cluster, kernel_zerosum)

# mask_output=True: reinsert NaN at originally-invalid pixels
r_masked = _nanconvolve(data_cluster, kernel_zerosum, mask_output=True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cmap = plt.cm.RdBu_r.copy()
cmap.set_bad('0.5')
vmax = np.nanmax(np.abs(gt_zero))

ax = axes[0]
im = ax.imshow(data_cluster, origin='lower', cmap=cmap, vmin=-3, vmax=3)
ax.set_title('Input data\n(NaN cluster shown in gray)', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

ax = axes[1]
im = ax.imshow(r_default, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
ax.set_title(f'Default (mask_output=False)\n'
             f'{np.sum(np.isnan(r_default))} NaN in output', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

ax = axes[2]
im = ax.imshow(r_masked, origin='lower', cmap=cmap, vmin=-vmax, vmax=vmax)
ax.set_title(f'mask_output=True\n'
             f'{np.sum(np.isnan(r_masked))} NaN in output', fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Zero-Sum Kernel — mask_output reinserts NaN at invalid pixels',
             fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

### 12b. `max_invalid_fraction` — extend mask to unreliable boundary pixels

When a large contiguous region is NaN/masked, pixels near the boundary have most of their kernel footprint overlapping invalid data. The renormalized convolution result at those pixels is unreliable — it's based on very few valid pixels.

`max_invalid_fraction` lets you flag those boundary pixels as NaN. For example:
- `max_invalid_fraction=0.1` → NaN if >10% of the kernel footprint is invalid
- `max_invalid_fraction=0.25` → NaN if >25% is invalid
- `max_invalid_fraction=0.5` → NaN if >50% is invalid

In [ ]:
fractions = [0.0, 0.1, 0.25, 0.5]

fig, axes = plt.subplots(2, len(fractions) + 1, figsize=(18, 7))
cmap = plt.cm.RdBu_r.copy()
cmap.set_bad('0.5')

for row, (kernel, gt, klabel) in enumerate([
    (kernel_norm, gt_norm, 'Normalized Kernel'),
    (kernel_zerosum, gt_zero, 'Zero-Sum Kernel'),
]):
    vmax = np.nanmax(np.abs(gt))

    # First column: mask_output=True only (no boundary extension)
    r = _nanconvolve(data_cluster, kernel, mask_output=True)
    ax = axes[row, 0]
    err = r - gt
    vlim = max(np.nanmax(np.abs(err[~np.isnan(err)])), 1e-10)
    im = ax.imshow(err, origin='lower', cmap=cmap, vmin=-vlim, vmax=vlim)
    n_nan = int(np.sum(np.isnan(r)))
    ax.set_title(f'mask_output only\n{n_nan} NaN', fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)
    if row == 0:
        ax.set_ylabel(klabel, fontsize=10)
    else:
        ax.set_ylabel(klabel, fontsize=10)

    # Remaining columns: increasing max_invalid_fraction
    for j, frac in enumerate(fractions):
        r = _nanconvolve(data_cluster, kernel,
                         max_invalid_fraction=frac)
        ax = axes[row, j + 1]
        err = r - gt
        valid = ~np.isnan(err)
        vlim = max(np.nanmax(np.abs(err[valid])) if valid.any() else 1,
                   1e-10)
        im = ax.imshow(err, origin='lower', cmap=cmap,
                       vmin=-vlim, vmax=vlim)
        n_nan = int(np.sum(np.isnan(r)))
        n_valid = int(np.sum(~np.isnan(r)))
        if valid.any():
            maxerr = np.nanmax(np.abs(err[valid]))
        else:
            maxerr = 0
        ax.set_title(f'max_invalid_frac={frac}\n'
                     f'{n_nan} NaN, max|err|={maxerr:.2e}', fontsize=8)
        plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Error vs Ground Truth — Effect of max_invalid_fraction\n'
             '(gray = NaN in output)', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

### 12c. Visualize the NaN regions themselves

Show which pixels get masked at each `max_invalid_fraction` threshold for the large NaN cluster.

In [ ]:
all_fractions = [None, 0.0, 0.1, 0.25, 0.5]
labels = ['mask_output\nonly', 'frac=0.0\n(any overlap)', 'frac=0.1',
          'frac=0.25', 'frac=0.5']

fig, axes = plt.subplots(1, len(all_fractions) + 1, figsize=(18, 3.5))

# Column 0: Input NaN locations
ax = axes[0]
ax.imshow(np.isnan(data_cluster), origin='lower', cmap='Reds',
          vmin=0, vmax=1)
ax.set_title(f'Input NaN\n{int(np.sum(np.isnan(data_cluster)))} pixels',
             fontsize=9)

# Remaining columns: NaN in output at each threshold
for i, (frac, label) in enumerate(zip(all_fractions, labels)):
    if frac is None:
        r = _nanconvolve(data_cluster, kernel_zerosum, mask_output=True)
    else:
        r = _nanconvolve(data_cluster, kernel_zerosum,
                         max_invalid_fraction=frac)
    nan_map = np.isnan(r)
    ax = axes[i + 1]
    ax.imshow(nan_map, origin='lower', cmap='Reds', vmin=0, vmax=1)
    n = int(np.sum(nan_map))
    ax.set_title(f'{label}\n{n} NaN pixels', fontsize=9)

fig.suptitle('NaN regions in output at different max_invalid_fraction '
             'thresholds\n(red = NaN)', fontsize=12, y=1.05)
fig.tight_layout()
plt.show()

### 12d. Effect on scattered NaN (all test cases)

For scattered NaN/mask, `max_invalid_fraction` has a smaller effect since very few pixels have significant kernel overlap with invalid data. This shows the tradeoff: a strict threshold masks more border pixels but also removes more valid data.

In [ ]:
frac_values = [None, 0.0, 0.1, 0.25, 0.5]
frac_labels = ['mask_output\nonly', 'frac=0.0', 'frac=0.1',
               'frac=0.25', 'frac=0.5']

fig, axes = plt.subplots(len(test_cases), len(frac_values),
                         figsize=(16, 3.2 * len(test_cases)))

for i, tc in enumerate(test_cases):
    d, m = tc['data'], tc['mask']
    for j, (frac, flabel) in enumerate(zip(frac_values, frac_labels)):
        if frac is None:
            r = _nanconvolve(d, kernel_zerosum, mask=m,
                             mask_output=True)
        else:
            r = _nanconvolve(d, kernel_zerosum, mask=m,
                             max_invalid_fraction=frac)

        n_nan = int(np.sum(np.isnan(r)))
        valid = ~np.isnan(r)
        err = r - gt_zero
        if valid.any():
            maxerr = np.max(np.abs(err[valid]))
        else:
            maxerr = 0.0

        ax = axes[i, j]
        nan_map = np.isnan(r)
        ax.imshow(nan_map, origin='lower', cmap='Reds', vmin=0, vmax=1)
        ax.set_title(f'{flabel}\n{n_nan} NaN, max|err|={maxerr:.2e}',
                     fontsize=7)
        if j == 0:
            ax.set_ylabel(tc['name'], fontsize=8)

fig.suptitle('Zero-Sum Kernel — NaN regions at different thresholds '
             '(all test cases)', fontsize=12, y=1.01)
fig.tight_layout()
plt.show()

### 12e. Summary table — NaN count and max error vs threshold

In [ ]:
rows_frac = []

for kernel, gt, klabel in [
    (kernel_norm, gt_norm, 'Normalized'),
    (kernel_zerosum, gt_zero, 'Zero-Sum'),
]:
    for tc in test_cases:
        d, m = tc['data'], tc['mask']
        for frac in [None, 0.0, 0.1, 0.25, 0.5]:
            if frac is None:
                r = _nanconvolve(d, kernel, mask=m, mask_output=True)
                flabel = 'mask_output only'
            else:
                r = _nanconvolve(d, kernel, mask=m,
                                 max_invalid_fraction=frac)
                flabel = f'frac={frac}'

            valid = ~np.isnan(r)
            n_nan = int(np.sum(~valid))
            n_valid = int(np.sum(valid))
            err = r - gt
            if valid.any():
                max_err = np.max(np.abs(err[valid]))
                mean_err = np.mean(np.abs(err[valid]))
            else:
                max_err = mean_err = 0.0

            rows_frac.append({
                'Kernel': klabel,
                'Test Case': tc['name'],
                'Threshold': flabel,
                '# Valid': n_valid,
                '# NaN': n_nan,
                'Max |err|': max_err,
                'Mean |err|': mean_err,
            })

df_frac = pd.DataFrame(rows_frac)

styled_frac = (df_frac.style
               .format({
                   'Max |err|': '{:.3e}',
                   'Mean |err|': '{:.3e}',
               })
               .set_caption('Effect of max_invalid_fraction on output '
                            'quality (stats over valid pixels only)')
               .set_table_styles([
                   {'selector': 'caption',
                    'props': [('font-size', '14px'),
                              ('font-weight', 'bold')]},
               ])
               )
styled_frac